In [11]:
import re
import os
import pandas as pd



csv_path = '../nlp/datasets/ru/labeled.csv'

if os.path.exists(csv_path):
    df1 = pd.read_csv(csv_path)
    df1['toxic'] = df1['toxic'].astype(int)
    print(f"First dataset opened. Samples: {len(df1)}")
else:
    print(f"File not found: {csv_path}.")


txt_path = '../nlp/datasets/ru/dataset.txt'


df1 = pd.read_csv(csv_path)


df1['toxic'] = df1['toxic'].astype(int)


parsed_data = []


with open(txt_path, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue


        match = re.match(r'(^__label__\S+)\s+(.*)', line)

        if match:
            labels_part = match.group(1)
            comment_text = match.group(2)


            if '__label__NORMAL' in labels_part and ',' not in labels_part:
                toxic = 0
            else:
                toxic = 1

            parsed_data.append({'comment': comment_text, 'toxic': toxic})


df2 = pd.DataFrame(parsed_data)

print(f"Second dataset opened. Samples: {len(df2)}")

df_final = pd.concat([df1, df2], ignore_index=True)

df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nMerged dataset samples: {len(df_final)}")
print("\nLabels distribution: (toxic=1, normal=0)")
print(df_final['toxic'].value_counts())


output_path = '../nlp/datasets/ru/merged_toxic_dataset.csv'
df_final.to_csv(output_path, index=False, encoding='utf-8')

print(f"\nMerged dataset saved at: {output_path}")

df_toxic = df_final[df_final['toxic'] == 1]
df_normal = df_final[df_final['toxic'] == 0]

min_class_size = min(len(df_toxic), len(df_normal))


df_normal_balanced = df_normal.sample(n=min_class_size, random_state=42)


df_toxic_balanced = df_toxic.sample(n=min_class_size, random_state=42)


df_balanced = pd.concat([df_toxic_balanced, df_normal_balanced], ignore_index=True)

df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Balanced dataset samples: {len(df_balanced)}")
print("\nLabels distribution: (toxic=1, normal=0) ")
print(df_balanced['toxic'].value_counts())

balanced_output_path = '../nlp/datasets/ru/balanced_toxic_dataset.csv'
df_balanced.to_csv(balanced_output_path, index=False, encoding='utf-8')

print(f"\nBalanced dataset saved at: {balanced_output_path}")

First dataset opened. Samples: 14412
Second dataset opened. Samples: 248290

Merged dataset samples: 262702

Labels distribution: (toxic=1, normal=0)
toxic
0    213271
1     49431
Name: count, dtype: int64

Merged dataset saved at: ../nlp/datasets/ru/merged_toxic_dataset.csv
Balanced dataset samples: 98862

Labels distribution: (toxic=1, normal=0) 
toxic
0    49431
1    49431
Name: count, dtype: int64

Balanced dataset saved at: ../nlp/datasets/ru/balanced_toxic_dataset.csv


In [4]:
!pip install -q transformers datasets evaluate accelerate

import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
import evaluate


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Code will be executed on: {device}") # cuda if on GPU

Code will be executed on: cpu


In [7]:

df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

total_len = len(df_balanced)


train_end = int(0.6 * total_len)
val_end = int(0.8 * total_len)


df_train = df_balanced.iloc[:train_end].copy()
df_val = df_balanced.iloc[train_end:val_end].copy()
df_test = df_balanced.iloc[val_end:].copy()


df_train = df_train[['comment', 'toxic']].rename(columns={'comment': 'text', 'toxic': 'label'})
df_val = df_val[['comment', 'toxic']].rename(columns={'comment': 'text', 'toxic': 'label'})
df_test = df_test[['comment', 'toxic']].rename(columns={'comment': 'text', 'toxic': 'label'})


dataset_train = Dataset.from_pandas(df_train)
dataset_val = Dataset.from_pandas(df_val)
dataset_test = Dataset.from_pandas(df_test)

print(f"--- Dataset split (60/20/20) ---")
print(f"Train:       {len(dataset_train)} samples (60%)")
print(f"Validation:  {len(dataset_val)} samples (20%)")
print(f"Test:        {len(dataset_test)} samples (20%)")

--- Dataset split (60/20/20) ---
Train:       59317 samples (60%)
Validation:  19772 samples (20%)
Test:        19773 samples (20%)


In [9]:
df_test.to_csv('../nlp/datasets/ru/ru_testset.csv', index=False, encoding='utf-8')
print("Test DataFrame saved to 'ru_testset.csv'")

df_train.to_csv('../nlp/datasets/ru/ru_trainset.csv', index=False, encoding='utf-8')
print("Train DataFrame saved to 'ru_trainset.csv'")

df_val.to_csv('../nlp/datasets/ru/ru_valset.csv', index=False, encoding='utf-8')
print("Validation DataFrame saved to 'ru_valset.csv'")

Test DataFrame saved to 'ru_testset.csv'
Train DataFrame saved to 'ru_trainset.csv'
Validation DataFrame saved to 'ru_valset.csv'


In [6]:
model_name = "DeepPavlov/rubert-base-cased-conversational"


tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):

    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)


tokenized_train = dataset_train.map(tokenize_function, batched=True)
tokenized_val = dataset_val.map(tokenize_function, batched=True)
tokenized_test = dataset_test.map(tokenize_function, batched=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.40M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/59317 [00:00<?, ? examples/s]

Map:   0%|          | 0/19772 [00:00<?, ? examples/s]

Map:   0%|          | 0/19773 [00:00<?, ? examples/s]

In [7]:

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)


metric_accuracy = evaluate.load("accuracy")
metric_f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    acc = metric_accuracy.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = metric_f1.compute(predictions=predictions, references=labels, average="binary")["f1"]

    return {"accuracy": acc, "f1": f1}

pytorch_model.bin:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: DeepPavlov/rubert-base-cased-conversational
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

In [8]:

training_args = TrainingArguments(
    output_dir="./rubert_toxic_model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=10,
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)


trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.190087,0.129117,0.964596,0.964069
2,0.072725,0.157436,0.968390,0.968468
3,0.044240,0.194550,0.967580,0.967605


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=11124, training_loss=0.08226736567503454, metrics={'train_runtime': 1598.82, 'train_samples_per_second': 111.301, 'train_steps_per_second': 6.958, 'total_flos': 1.170521885309184e+16, 'train_loss': 0.08226736567503454, 'epoch': 3.0})

In [9]:
import os
from google.colab import files


model_export_path = "/content/my_toxic_rubert"
trainer.save_model(model_export_path)
tokenizer.save_pretrained(model_export_path)


!zip -r /content/my_toxic_rubert.zip /content/my_toxic_rubert


files.download('/content/my_toxic_rubert.zip')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  adding: content/my_toxic_rubert/ (stored 0%)
  adding: content/my_toxic_rubert/tokenizer.json (deflated 73%)
  adding: content/my_toxic_rubert/tokenizer_config.json (deflated 42%)
  adding: content/my_toxic_rubert/model.safetensors (deflated 8%)
  adding: content/my_toxic_rubert/config.json (deflated 56%)
  adding: content/my_toxic_rubert/training_args.bin (deflated 53%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
test_results = trainer.evaluate(eval_dataset=tokenized_test)


print("\nResults on a test set:")
print(f"Accuracy: {test_results['eval_accuracy']:.4f}")
print(f"F1-Score: {test_results['eval_f1']:.4f}")


Results on a test set:
Accuracy: 0.9680
F1-Score: 0.9683


In [11]:
import pandas as pd
import numpy as np

predictions_output = trainer.predict(tokenized_test)


raw_predictions = predictions_output.predictions
predictions = np.argmax(raw_predictions, axis=-1)


true_labels = predictions_output.label_ids


df_analysis = df_test.copy()
df_analysis['predicted_label'] = predictions
df_analysis['is_correct'] = df_analysis['label'] == df_analysis['predicted_label']


df_analysis['true_toxic_text'] = df_analysis['label'].map({1: 'toxic', 0: 'normal'})
df_analysis['pred_toxic_text'] = df_analysis['predicted_label'].map({1: 'toxic', 0: 'normal'})



In [12]:
df_analysis[['text', 'true_toxic_text', 'pred_toxic_text', 'is_correct']].head(10)

,text,true_toxic_text,pred_toxic_text,is_correct
79089,"ну пять рублей, ну 98 год. и?",normal,normal,True
79090,народу затуманили мозги...кругом были смотрящи...,toxic,normal,False
79091,я что то не пойму за что поддерживать? мне воо...,toxic,toxic,True
79092,продам браслет. подьебка.первому клиенту минет...,toxic,toxic,True
79093,мой как говорил дед еби кривых сорбатых косых ...,toxic,toxic,True
79094,И леваки этого Александра грохнули. Ебанутые.\n,toxic,toxic,True
79095,если деньги завтра отправлю,normal,normal,True
79096,"Ты опять выходишь на связь, шизик, тебя уже ис...",toxic,toxic,True
79097,ты долбоеб если так рассуждаешь! мне больше те...,toxic,toxic,True
79098,расстрелять на хуй....,toxic,toxic,True


In [13]:

errors = df_analysis[df_analysis['is_correct'] == False]
print(f"Errors: {len(errors)} from {len(df_analysis)}")
errors[['text', 'true_toxic_text', 'pred_toxic_text']].sample(min(5, len(errors)))

Errors: 633 from 19773


,text,true_toxic_text,pred_toxic_text
88923,"в крови ефремова обнаружены алкоголь, кокаин, ...",normal,toxic
86106,"правильно надо вернуть смертную казнь ,вообще ...",normal,toxic
90700,чем кололи? подстилка должна быть чистой.,normal,toxic
83066,оборзевшие твари👎,toxic,normal
94703,"Подъехал на своей машине к месту, освобождаешь...",toxic,normal


In [14]:
kaka = {}
kaka["text"] = ['да иди ты лесом вдоль дорожки']
input = Dataset.from_dict(kaka)

transformed_input = input.map(tokenize_function, batched=True)

print(kaka.keys())
print(tokenized_test)
answer = trainer.predict(transformed_input)
print(answer)


Map:   0%|          | 0/1 [00:00<?, ? examples/s]

dict_keys(['text'])
Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 19773
})


PredictionOutput(predictions=array([[ 3.2382812, -2.7460938]], dtype=float32), label_ids=None, metrics={'test_runtime': 0.0919, 'test_samples_per_second': 10.883, 'test_steps_per_second': 10.883})


In [15]:
import numpy as np


logits = answer.predictions


predicted_class_id = np.argmax(logits, axis=-1)[0]


exp_logits = np.exp(logits - np.max(logits))
probabilities = exp_logits / np.sum(exp_logits, axis=-1, keepdims=True)
confidence = probabilities[0][predicted_class_id]


labels_map = {0: "normal", 1: "toxic"}
final_label = labels_map[predicted_class_id]


print(f"Text: '{kaka['text'][0]}'")
print(f"Answer of the model: {final_label}")
print(f"Confidence of the model: {confidence:.2%}")

Text: 'да иди ты лесом вдоль дорожки'
Answer of the model: normal
Confidence of the model: 99.75%
